Seeing models as computation graphs

In [2]:
import torch

The following code implements the forward pass (prediction step) of a simple logistic regression classifier, which can be seen as a single-layer neural network, returning a score between 0 and 1 that is compared to the true class label (0 or 1) when computing the loss:

In [3]:
import torch.nn.functional as F

y = torch.tensor([1.0])  # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0])  # bias unit

z = x1 * w1 + b          # net input
a = torch.sigmoid(z)     # activation & output

loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


The point of this example is not to implement a logistic regression classifier but rather to illustrate how we can think of a sequence of computations as a computation graph

Automatic differentiation made easy:

In [4]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)


In [5]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [6]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


Implementing multilayer neural networks:

In [3]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

We can then instantiate a new neural network object as follows:

In [4]:
model = NeuralNetwork(50, 3)

In [5]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Next, let’s check the total number of trainable parameters of this model:

In [6]:
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [7]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-3.1942e-02, -1.0708e-01,  6.6184e-02,  ...,  9.0159e-03,
          1.3544e-01,  6.4164e-05],
        [ 1.0732e-01, -1.3205e-01,  1.1988e-01,  ..., -4.0759e-02,
         -1.3275e-01,  1.2299e-01],
        [-5.0258e-02,  9.1825e-02,  5.4190e-02,  ..., -9.1688e-02,
         -1.3466e-01,  5.5887e-02],
        ...,
        [-1.3460e-01, -1.3701e-01,  2.1229e-02,  ...,  9.0245e-02,
          6.4646e-02, -1.3561e-01],
        [ 1.0255e-01,  8.9620e-02,  7.0771e-02,  ..., -2.0833e-02,
          8.1968e-02, -7.9224e-02],
        [ 1.2582e-01,  7.3758e-02,  1.0128e-01,  ...,  6.2156e-02,
         -1.2936e-01,  3.0667e-02]], requires_grad=True)


In [8]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])
